In [ ]:
import numpy as np
import copy 

from fair.stats.survey import Corpus, SingleTopicSurvey
from fair.agent import LegacyStudent
from fair.allocation import general_yankee_swap_E, round_robin, serial_dictatorship, integer_linear_program
from fair.optimization import StudentAllocationProgram
from fair.metrics import utilitarian_welfare, nash_welfare
from fair.envy import EF_violations_reponses
from matplotlib import pyplot as plt
from sklearn.decomposition import PCA

import qsurvey

NUM_RAND_SAMP = 20
NUM_SUB_KERNELS = 3
SAMPLE_PER_STUDENT = 10
SPARSE = False
PLOT = True
seed = 0
RNG = np.random.default_rng(seed)
pref_thresh = 100

In [ ]:
status_color_map = {
    1: "lightsteelblue",
    2: "blue",
    3: "forestgreen",
    4: "darkkhaki",
    5: "darkorange",
    6: "red",
}
status_max_course_map = {
    1: 6,
    2: 6,
    3: 6,
    4: 6,
    5: 4,
    6: 4,
}
status_crs_prefix_map = {
    1: ["1", "2", "3"],
    2: ["1", "2", "3", "4"],
    3: ["1", "2", "3", "4", "5"],
    4: ["2", "3", "4", "5", "6"],
    5: ["5", "6"],
    6: ["5", "6"],
}

# survey_file = "../resources/survey_data.csv"
survey_file = "../resources/random_survey.csv"
schedule_file = "../resources/anonymized_courses.xlsx"
mapping_file = "../resources/survey_column_mapping.csv"

In [ ]:
mp = qsurvey.QMapper(mapping_file)
qd = qsurvey.QSchedule(schedule_file)
crs_sec_cap_map = qd.capacities()
qs = qsurvey.QSurvey(survey_file, mp, list(crs_sec_cap_map.keys()))
course_map = mp.mapping(qs.all_courses)
all_courses = [crs for crs in course_map.keys()]
features = mp.features(course_map)
course, slot, weekday, section = features
schedule = mp.schedule(course_map, crs_sec_cap_map, features)
students, responses, statuses = qs.students(
    course_map, all_courses, features, schedule, status_max_course_map, pref_thresh, SPARSE
)
student_status_map = {students[i]: status for i, status in enumerate(statuses)}
student_resp_map = {students[i]: response for i, response in enumerate(responses)}
course_cap_map = {
    crs: crs_sec_cap_map[course_map[crs]["course num"]][int(course_map[crs]["section"])]
    for crs in all_courses
}
students = [
    student for student in students if len(student.student.preferred_courses) > 0
]

In [ ]:
# Build small example:
for sche in schedule:
    sche.capacity = 0

for i in range(51,55):
    schedule[i].capacity=1

students = students[:4]

c = np.vstack([student_resp_map[student] for student in students])-1
c[1,53]=3
c[:,51:55]

In [ ]:
c = np.vstack([student_resp_map[student] for student in students])-1

In [ ]:
X_ILP = integer_linear_program(students, schedule, valuations= c)
X_ILP[51:55, :]

In [ ]:
X_SD = serial_dictatorship(students, schedule, c)
X_SD[51:55,:]

In [ ]:
X_RR = round_robin(students, schedule, c)
X_RR[51:55,:]

In [ ]:
X_YS, _, _ = general_yankee_swap_E(students, schedule, valuations=c)
X_YS[51:55, :]

In [ ]:
current_utilities = np.diag(np.dot(c,X_ILP))
print(current_utilities)
print(EF_violations_reponses(X_ILP, students, schedule, student_status_map,c))
print(utilitarian_welfare(X_ILP, students, schedule, current_utilities))
print(nash_welfare(X_ILP, students, schedule, current_utilities))

In [ ]:
current_utilities = np.diag(np.dot(c,X_SD))
print(current_utilities)
print(EF_violations_reponses(X_SD, students, schedule, student_status_map,c))
print(utilitarian_welfare(X_SD, students, schedule, current_utilities))
print(nash_welfare(X_SD, students, schedule, current_utilities))

In [ ]:
current_utilities = np.diag(np.dot(c,X_RR))
print(current_utilities)
print(EF_violations_reponses(X_RR, students, schedule, student_status_map,c))
print(utilitarian_welfare(X_RR, students, schedule, current_utilities))
print(nash_welfare(X_RR, students, schedule, current_utilities))

In [ ]:
current_utilities = np.diag(np.dot(c,X_YS))
print(current_utilities)
print(EF_violations_reponses(X_YS, students, schedule, student_status_map,c))
print(utilitarian_welfare(X_YS, students, schedule, current_utilities))
print(nash_welfare(X_YS, students, schedule, current_utilities))